## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-16-50-49 +0000,nyt,How a Logger From Maine Toppled a Giant of the...,https://www.nytimes.com/2026/11/04/us/politics...
1,2026-11-04-16-48-31 +0000,nyt,Trump and Republicans Begin Pointing Fingers A...,https://www.nytimes.com/2026/11/04/us/politics...
2,2026-11-04-16-47-01 +0000,startribune,Amy Klobuchar and Peggy Flanagan triumph as De...,https://www.startribune.com/live-minnesota-202...
3,2026-11-04-16-47-00 +0000,wsj,Letitia James Steps Back From Probe Into Alleg...,https://www.wsj.com/us-news/law/letitia-james-...
4,2026-11-04-16-46-25 +0000,nyt,An 80-Cent Plastic Pouch Is Saving Thousands o...,https://www.nytimes.com/2026/11/03/health/mate...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2426/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
5,trump,54
64,democrats,49
54,race,45
159,election,43
78,midterm,36


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
384,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,264
162,2026-11-04-11-34-05 +0000,chitri,Democrats win the US House and Senate as voter...,https://www.chicagotribune.com/2026/11/04/poll...,231
281,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...,210
251,2026-11-04-05-28-25 +0000,chitri,"Iowa governor race: Rob Sand wins election, fl...",https://www.chicagotribune.com/2026/11/03/rob-...,203
45,2026-11-04-15-51-00 +0000,wsj,The Democrats’ big victories in the House and ...,https://www.wsj.com/politics/elections/trump-p...,203


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
384,264,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
281,210,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...
124,118,2026-11-04-12-54-34 +0000,cbc,Iranian officials welcome U.S. midterm results...,https://www.cbc.ca/news/world/iran-united-stat...
286,103,2026-11-04-04-33-12 +0000,chitri,Lake County Sheriff John Idleburg declares vic...,https://www.chicagotribune.com/2026/11/03/lake...
41,96,2026-11-04-15-55-00 +0000,wsj,Elon Musk’s political fortunes have hit a blue...,https://www.wsj.com/politics/elections/elon-mu...
295,75,2026-11-04-04-08-32 +0000,latimes,Xavier Becerra makes history as California's f...,https://www.latimes.com/california/story/2026-...
134,70,2026-11-04-12-36-00 +0000,wsj,Americans’ long-simmering frustrations over th...,https://www.wsj.com/politics/elections/were-ju...
291,66,2026-11-04-04-28-16 +0000,latimes,Ugarte takes early lead for South L.A. City Co...,https://www.latimes.com/california/story/2026-...
346,65,2026-11-04-01-00-20 +0000,chitri,Voters usher in new era with state Rep. La Sha...,https://www.chicagotribune.com/2026/11/03/illi...
23,57,2026-11-04-16-25-55 +0000,nyt,Mike Johnson Won’t Seek Re-Election as House L...,https://www.nytimes.com/2026/11/04/us/mike-joh...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
